# Lecture 17 – Data 100, Spring 2026

[Acknowledgments Page](https://ds100.org/acknowledgements/)

In [ ]:
import json
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

import warnings
warnings.filterwarnings('ignore')


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import root_mean_squared_error

sns.set()

In this lecture, we will work with the `vehicles` dataset.

In [ ]:
vehicles = sns.load_dataset("mpg").rename(columns={"horsepower":"hp"}).dropna().sort_values("hp")
vehicles.head()

We will attempt to predict a car's `"mpg"` from transformations of its `"hp"`.

In [ ]:
X = vehicles[["hp"]]
X["hp^2"] = vehicles["hp"]**2
X["hp^3"] = vehicles["hp"]**3
X["hp^4"] = vehicles["hp"]**4

Y = vehicles["mpg"]

In [ ]:
X.head()

## Test Set

To perform a train-test split, we can use the `train_test_split` function of the `sklearn.model_selection` module.

In [ ]:
from sklearn.model_selection import train_test_split

# `test_size` specifies the proportion of the full dataset that should be allocated to testing.
# `random_state` makes our results reproducible for educational purposes.
# shuffle is True by default and randomizes the data before splitting.

X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y,
    test_size=0.2, 
    random_state=100, 
    shuffle=True
)

print(f"Size of full dataset: {X.shape[0]} points")
print(f"Size of training set: {X_train.shape[0]} points")
print(f"Size of test set: {X_test.shape[0]} points")

We then fit the model using the training set...

In [ ]:
import sklearn.linear_model as lm

model = lm.LinearRegression()

model.fit(X_train, Y_train)

model.coef_

...and evaluate its performance by making predictions on the test set. Notice that the model performs more poorly on the test data it did not encounter during training.

In [ ]:
from sklearn.metrics import mean_squared_error

train_error = mean_squared_error(Y_train, model.predict(X_train))
test_error = mean_squared_error(Y_test, model.predict(X_test))

print(f"Training error: {train_error}")
print(f"Test error: {test_error}")

## Validation Set

To estimate model performance on unseen data, and then *use* this information to finetune the model, we introduce a validation set. You can imagine this as us splitting the training set into a validation set and a "mini" training set.

In [ ]:
# Split X_train further into X_train_mini and X_val.
X_train_mini, X_val, Y_train_mini, Y_val = train_test_split(X_train, Y_train, test_size=0.2, random_state=100)

print(f"Size of original training set: {X_train.shape[0]} points")
print(f"Size of mini training set: {X_train_mini.shape[0]} points")
print(f"Size of validation set: {X_val.shape[0]} points")

In the cell below, we repeat the experiment from Lecture 14: fit several models of increasing complexity, then compute their errors. Here, we find the model's errors on the **validation set** to understand how model complexity influences performance on unseen data.

In [ ]:
fig, ax = plt.subplots(1, 3, dpi=200, figsize=(12, 3))

# iterate over polynomial orders 2, 3, and 4
for order in [2, 3, 4]:
    model = lm.LinearRegression()
    model.fit(X_train_mini.iloc[:, :order], Y_train_mini)
    val_predictions = model.predict(X_val.iloc[:, :order])
    
    output = X_val.iloc[:, :order]
    output["y_hat"] = val_predictions
    output = output.sort_values("hp")
    
    ax[order-2].scatter(X_val["hp"], Y_val, edgecolor="white", lw=0.5)
    ax[order-2].plot(output["hp"], output["y_hat"], "tab:red")
    ax[order-2].set_title(f"Model with degree {order}")
    ax[order-2].set_xlabel("hp")
    ax[order-2].set_ylabel("mpg")
    ax[order-2].annotate(f"Validation MSE: {np.round(mean_squared_error(Y_val, val_predictions), 3)}", (90, 30))

plt.subplots_adjust(wspace=0.3);

Let's repeat this process:

1. Fit an degree-x model to the mini training set
2. Evaluate the fitted model's MSE when making predictions on the validation set

We use the model's performance on the validation set as a guide to selecting the best combination of features. We are not limited in the number of times we use the validation set – we just never use this set to fit the model.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

def fit_model_dataset(degree):
    pipelined_model = Pipeline([
        ('polynomial_transformation', PolynomialFeatures(degree)),
        ('linear_regression', lm.LinearRegression())    
    ])

    pipelined_model.fit(X_train_mini[["hp"]], Y_train_mini)
    return mean_squared_error(Y_val, pipelined_model.predict(X_val[["hp"]]))

errors = [fit_model_dataset(degree) for degree in range(0, 18)]
MSEs_and_k = pd.DataFrame({"k": range(0, 18), "MSE": errors})

plt.figure(dpi=120)
plt.plot(range(0, 18), errors)
plt.xlabel("Model Complexity (degree of polynomial)")
plt.ylabel("Validation MSE")
plt.xticks(range(0, 18));

In [ ]:
MSEs_and_k.rename(columns={"k":"Degree"}).set_index("Degree")

From this **model selection** process, we might choose to create a model with degree 8.

In [ ]:
print(f'Polynomial degree with lowest validation error: {MSEs_and_k.sort_values("MSE").head(1)["k"].values}')

After this choice has been finalized, and we are completely finished with the model design process, we finally assess model performance on the test set. We typically use the entire training set (both the "mini" training set and validation set) to fit the final model.

In [ ]:
# Update our training and test sets to include all polynomial features between 5 and 9
for degree in range(5, 9):
    X_train[f"hp^{degree}"] = X_train["hp"]**degree
    X_test[f"hp^{degree}"] = X_test["hp"]**degree

final_model = lm.LinearRegression()
final_model.fit(X_train, Y_train)

print(f"Test MSE of final model: {mean_squared_error(Y_test, final_model.predict(X_test))}")

## Cross-Validation

The validation set gave us an opportunity to understand how the model performs on a **single** set of unseen data. The specific validation set we drew was fixed – we used the same validation points every time.

It's possible that we may have, by random chance, selected a set of validation points that was *not* representative of other unseen data that the model might encounter (for example, if we happened to have selected all outlying data points for the validation set).

Different train/validation splits lead to different validation errors:

In [ ]:
for i in range(1, 4):
    X_train_mini, X_val, Y_train_mini, Y_val = train_test_split(X_train, Y_train, test_size=0.2)
    model = lm.LinearRegression()
    model.fit(X_train_mini, Y_train_mini)
    y_hat = model.predict(X_val)
    print(f"Val error from train/validation split #{i}: {mean_squared_error(y_hat, Y_val)}")

To apply cross-validation, we use the `KFold` class of `sklearn.model_selection`. `KFold` will return the indices of each cross-validation fold. Then, we iterate over each of these folds to designate it as the validation set, while training the model on the remaining folds.

In [ ]:
from sklearn.model_selection import KFold
np.random.seed(25) # Ensures reproducibility of this notebook

# n_splits sets the number of folds to create
kf = KFold(n_splits=5, shuffle=True)
validation_errors = []

for train_idx, valid_idx in kf.split(X_train):
    # Split the data
    split_X_train, split_X_valid = X_train.iloc[train_idx], X_train.iloc[valid_idx]
    split_Y_train, split_Y_valid = Y_train.iloc[train_idx], Y_train.iloc[valid_idx]

    # Fit the model on the training split
    model.fit(split_X_train, split_Y_train)

    error = mean_squared_error(model.predict(split_X_valid), split_Y_valid)

    validation_errors.append(error)

print(f"Cross-validation error: {np.mean(validation_errors)}")

## Regularization

### L1 (LASSO) Regularization

To apply L1 regularization, we use the `Lasso` model class of `sklearn`. `Lasso` functions just like `LinearRegression`. The difference is that now the model will apply a *regularization penalty*. We specify the strength of regularization using the `alpha` parameter, which is equivalent to $\frac{1}{\lambda}$ from our objective function formulation.

In [ ]:
import sklearn.linear_model as lm

lasso_model = lm.Lasso(alpha=0.1) # In sklearn, alpha represents the lambda hyperparameter
lasso_model.fit(X_train, Y_train) 

lasso_model.coef_

To increase the strength of regularization (decrease model complexity), we increase the $\lambda$ hyperparameter by changing `alpha`.

In [ ]:
lasso_model_large_lambda = lm.Lasso(alpha=10)
lasso_model_large_lambda.fit(X_train, Y_train) 

lasso_model_large_lambda.coef_

Notice that these model coefficients are very small (some are effectively 0). This reflects L1 regularization's tendency to set the parameters of unimportant features to 0. We can use this in **feature selection**.

The features in our dataset are on wildly different numerical scales. To see this, compare the values of `hp` to the values of `hp^8`.

In [ ]:
X_train.head()

In order for the feature `hp` to contribute in any meaningful way to the model, LASSO is "forced" to allocate disproportionately much of its parameter "budget" towards assigning a large value to the model parameter for `hp`. Notice how the parameter for `hp` is much, much greater in magnitude than the parameter for `hp^8`.

In [ ]:
pd.DataFrame({"Feature":X_train.columns, "Parameter":lasso_model.coef_})

We typically **scale** data before regularization such that all features are measured on the same numeric scale. One way to do this is by **standardizing** the data such that it has mean 0 and standard deviation 1.

In [ ]:
# Center the data to have mean 0
X_train_centered = X_train - X_train.mean() 

# Scale the centered data to have SD 1
X_train_standardized = X_train_centered/X_train_centered.std()

X_train_standardized.head()

When we re-fit a LASSO model, the coefficients are no longer as uneven in magnitude as they were before.

In [ ]:
lasso_model_scaled = lm.Lasso(alpha=0.1)
lasso_model_scaled.fit(X_train_standardized, Y_train)
lasso_model_scaled.coef_

### L2 (Ridge) Regression

We perform ridge regression using `sklearn`'s `Ridge` class.

In [ ]:
ridge_model = lm.Ridge(alpha=0.1)
ridge_model.fit(X_train_standardized, Y_train)

ridge_model.coef_

## Example: predicting how much bleaching occurs in coral reefs

![Infographic by NOAA titled 'Coral Bleaching'. Healthy coral: Coral and algae depend on each other to survive. Stressed coral: If stressed, algae leaves the coral. Beached Coral: Coral is left bleached and vulnerable. Change in ocean temperature, runoff and pollution, overexposure to sunlight, and extreme low tides cause coral bleaching. ](https://oceanservice.noaa.gov/facts/coralbleaching-large.jpg)

## Data

In [ ]:
reefs = pd.read_csv('reef.csv')

# Only do predictions for reefs with bleaching
reefs = reefs[reefs['Average_bleaching'] > 0]

# This tells pandas to show all the columns of the dataframe, just within this block
with pd.option_context('display.max_columns', None):
    display(reefs.sample(3))

In [ ]:
X_cols = ['Year', 'ClimSST', 'Temperature_Kelvin', 'SSTA', 'TSA', 'Diversity', 'rate_of_SST_change', 'Longitude.Degrees', 'Latitude.Degrees']
y_col = 'Average_bleaching'

# This is not generally a good practice: why not? What might you do instead?
reefs = reefs.dropna(subset = X_cols + [y_col], how='any')

X = reefs[X_cols]
y = reefs[y_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=12345)

In [ ]:
X_train.head(3)

In [ ]:
y_train.head(3)

### Understanding LASSO

### For reference: using OLS

In [ ]:
ols_model = LinearRegression()
ols_model.fit(X_train, y_train)
ols_model.coef_

In [ ]:
print(root_mean_squared_error(ols_model.predict(X_val), y_val))

### First attempt: using LASSO

In [ ]:
lasso_01 = Lasso(alpha=0.1)
lasso_01.fit(X_train, y_train)
lasso_01.coef_

In [ ]:
print(root_mean_squared_error(lasso_01.predict(X_val), y_val))

Let's decrease the budget (i.e., increase $\lambda$) and see what happens:

In [ ]:
lasso_1 = Lasso(alpha=1)
lasso_1.fit(X_train, y_train)
lasso_1.coef_

In [ ]:
print(root_mean_squared_error(lasso_1.predict(X_val), y_val))

Now, some of the values we got back are 0! But this isn't a fair way to allocate our budget, because a small change in the $\theta$ value for `Diversity` (how many species are in the coral) can dramatically shift our output, just because the values in that column tend to be much larger. 

If we want to fairly allocate our budget across the different $\theta$ values, we need to **standardize** our inputs first: this means subtracting the mean and dividing by the standard deviation (more on these later).

You've seen these operations before in Data 8!

In [ ]:
X_train_su = (X_train - X_train.mean())/X_train.std()

# Be careful: make sure you understand why we're using X_train.mean() 
# instead of X_val.mean() or X.mean()!
X_val_su = (X_val - X_train.mean()) / X_train.std()

In [ ]:
lasso_01_su = Lasso(alpha=0.1)
lasso_01_su.fit(X_train_su, y_train)
lasso_01_su.coef_

In [ ]:
print(root_mean_squared_error(lasso_01_su.predict(X_val_su), y_val))

In [ ]:
lasso_1_su = Lasso(alpha=1)
lasso_1_su.fit(X_train_su, y_train)
lasso_1_su.coef_

In [ ]:
print(root_mean_squared_error(lasso_1_su.predict(X_val_su), y_val))

## How do different values of $\lambda$ (or `alpha` in `sklearn`) affect the results?

In [ ]:
alphas = np.logspace(-3, 2, 100)
coeffs_lasso = []
lasso_val_errors = []
lasso_train_errors = []
for alpha in alphas:
    model = Lasso(alpha=alpha)
    model.fit(X_train_su, y_train)
    coeffs_lasso.append(model.coef_)
    lasso_val_errors.append(root_mean_squared_error(model.predict(X_val_su), y_val))
    lasso_train_errors.append(root_mean_squared_error(model.predict(X_train_su), y_train))
    

#coefficient_names = [col_map[c] for c in X_cols]
coefficient_names = X_cols

plt.plot(alphas, coeffs_lasso, label=coefficient_names)
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'Coefficient $\theta$')
plt.semilogx()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.title(r'LASSO coefficients changing with $\lambda$');

In [ ]:
plt.plot(alphas, lasso_val_errors, label='Validation RMSE')
# Uncomment this line to see how the training error changes
# plt.plot(alphas, lasso_train_errors, label='Training RMSE')
plt.title('LASSO error as a function of $\lambda$')
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'RMSE');
plt.legend()
plt.semilogx();

In [ ]:
alphas = np.logspace(0, 7, 30)
coeffs_ridge = []
ridge_val_errors = []
for alpha in alphas:
    model = Ridge(alpha=alpha)
    model.fit(X_train_su, y_train)
    coeffs_ridge.append(model.coef_)
    ridge_val_errors.append(root_mean_squared_error(model.predict(X_val_su), y_val))

#coefficient_names = [col_map[c] for c in X_cols]
coefficient_names = X_cols

plt.plot(alphas, coeffs_ridge, label=coefficient_names)
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'Coefficient $\theta$')
plt.semilogx()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.title(r'Ridge coefficients changing with $\lambda$');

In [ ]:
plt.plot(alphas, ridge_val_errors)
plt.title('Ridge validation error as a function of $\lambda$')
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'Validation RMSE');
plt.semilogx();

Bonus exercise: how would you modify the code above to use cross-validation error (CV error) instead of just validation set error?

## Bonus: using LASSO and Ridge with even more predictors and OHE in a scikit-learn `Pipeline`

In [ ]:
X_cols = [
    'Ocean', 'Year', 'ClimSST', 'Temperature_Kelvin', 'SSTA', 'TSA', 'Diversity', 
    'rate_of_SST_change', 'Longitude.Degrees', 'Latitude.Degrees'
]
y_col = 'Average_bleaching'
numeric_X_cols = X_cols[1:]

In [ ]:
X = reefs.loc[:, X_cols]
y = reefs[y_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=12345)

In [ ]:
def make_linear_model_pipeline(model_type):

    pl = Pipeline([
        (
            "Feature Engineering", 
            ColumnTransformer(  
                [
                    ("OHE", OneHotEncoder(), ["Ocean"]),
                    ("Scale", StandardScaler(), numeric_X_cols),
                ], 
                remainder="passthrough")
        ),
        ("Model", model_type())
    ]) 
    return pl

In [ ]:
alphas = np.logspace(-5, 1, 30)
coeffs_lasso_ohe = []
val_err_lasso_ohe = []
for alpha in alphas:
    pipe = make_linear_model_pipeline(Lasso)
    pipe.set_params(Model__alpha=alpha)
    pipe.fit(X_train, y_train)
    coeffs_lasso_ohe.append(pipe['Model'].coef_)
    val_err_lasso_ohe.append(root_mean_squared_error(pipe.predict(X_val), y_val))

coefficient_names = pipe[:-1].get_feature_names_out()

In [ ]:
plt.plot(alphas, coeffs_lasso_ohe, label=coefficient_names)
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'Coefficient $\theta$')
plt.semilogx()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5));
plt.title(r'LASSO coefficients changing with $\lambda$');

In [ ]:
plt.plot(alphas, val_err_lasso_ohe, label='Validation RMSE')
plt.title('LASSO error as a function of $\lambda$')
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'RMSE');
plt.semilogx();

In [ ]:
alphas = np.logspace(-1, 5, 30)
coeffs_ridge_ohe = []
val_err_ridge_ohe = []
for alpha in alphas:
    pipe = make_linear_model_pipeline(Ridge)
    pipe.set_params(Model__alpha=alpha)
    pipe.fit(X_train, y_train)
    coeffs_ridge_ohe.append(pipe['Model'].coef_)
    val_err_ridge_ohe.append(root_mean_squared_error(pipe.predict(X_val), y_val))

coefficient_names = pipe[:-1].get_feature_names_out()

In [ ]:
plt.plot(alphas, coeffs_ridge_ohe, label=coefficient_names)
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'Coefficient $\theta$')
plt.semilogx()
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5));
plt.title(r'Ridge coefficients changing with $\lambda$');

In [ ]:
plt.plot(alphas, ridge_val_errors)
plt.title('Ridge validation error as a function of $\lambda$')
plt.xlabel(r'$\lambda$ (log scale)')
plt.ylabel(r'Validation RMSE');
plt.semilogx();

Bonus question: How would you improve the last two coefficient visualizations to better communicate their intended message?
